# Notebook 04: HSI Weighted Overlay, Statistics & Map layout

This notebook calculates the final Habitat Suitability Index (HSI) using a Weighted Overlay, classifies HSI into suitability categories, computes spatial extent statistics, and plots a publication-ready final map.

## Objectives:
1. Calculate Weighted Overlay (HSI).
2. Classify HSI into 5 category classes.
3. Compute area statistics ($$km^2$$ and percentage) and export to CSV.
4. Plot the final publication map with a scale bar, north arrow, coordinate grid, and legend.

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import rasterio

# Configure paths
NOTEBOOK_DIR = Path(os.getcwd())
PROJECT_ROOT = NOTEBOOK_DIR.parent
sys.path.append(str(PROJECT_ROOT))

from src.suitability import calculate_hsi, classify_hsi, calculate_statistics
from src.visualization import plot_publication_map, plot_statistics_chart

print('Environment ready!')

## 1. HSI Weighted Overlay Model

We apply the weights: 
- NDVI: 30%
- Distance to Water: 25%
- LULC: 20%
- Slope: 15%
- DEM (Elevation): 10%

$$\text{HSI} = 0.30 \times \text{NDVI} + 0.25 \times \text{DistanceToWater} + 0.20 \times \text{LULC} + 0.15 \times \text{Slope} + 0.10 \times \text{Elevation}$$

In [ ]:
reclass_dir = PROJECT_ROOT / 'data' / 'processed' / 'Reclassified'
output_dir = PROJECT_ROOT / 'data' / 'processed' / 'Output'
output_dir.mkdir(parents=True, exist_ok=True)

reclass_paths = {
    'ndvi': reclass_dir / 'NDVI_Reclass.tif',
    'dem': reclass_dir / 'DEM_Reclass.tif',
    'slope': reclass_dir / 'Slope_Reclass.tif',
    'distance': reclass_dir / 'Distance_Reclass.tif',
    'lulc': reclass_dir / 'LULC_Reclass.tif'
}

reclassed_data = {}
for name, path in reclass_paths.items():
    with rasterio.open(path) as src:
        reclassed_data[name] = src.read(1)
        out_profile = src.profile.copy()

weights = {
    'ndvi': 0.30,
    'distance': 0.25,
    'lulc': 0.20,
    'slope': 0.15,
    'dem': 0.10
}

hsi = calculate_hsi(reclassed_data, weights)
hsi_class = classify_hsi(hsi)

# Mask out water bodies (class 0 in Dynamic World LULC) from the suitability calculations
with rasterio.open(PROJECT_ROOT / 'data' / 'processed' / 'Cleaned' / 'DynamicWorld_Clean.tif') as src:
    lulc_clean = src.read(1)

hsi[lulc_clean == 0] = np.nan
hsi_class[lulc_clean == 0] = np.nan

# Save GeoTIFFs
hsi_path = output_dir / 'Habitat_Suitability_Index.tif'
class_path = output_dir / 'Habitat_Suitability_Class.tif'

out_profile.update(dtype='float32', nodata=np.nan, compress='lzw')

with rasterio.open(hsi_path, 'w', **out_profile) as dst:
    dst.write(hsi, 1)
with rasterio.open(class_path, 'w', **out_profile) as dst:
    dst.write(hsi_class, 1)

print('HSI maps saved!')

## 2. Area Statistics

We calculate the area in square kilometers for each suitability class (assuming 10m pixel resolution).

In [ ]:
stats_df = calculate_statistics(hsi_class, resolution=10.0)
stats_df.to_csv(output_dir / 'Habitat_Suitability_Statistics.csv', index=False)
stats_df

## 3. Publication-Grade Map and Chart

We call the visualization functions to generate high-resolution figures.

In [ ]:
aoi_path = PROJECT_ROOT / 'data' / 'raw' / 'AOI' / 'Corbett_AOI.shp'
map_jpg = output_dir / 'Corbett_Habitat_Suitability_Map.png'
chart_jpg = output_dir / 'Habitat_Suitability_Statistics_Chart.png'

# Generate map
plot_publication_map(class_path, aoi_path, map_jpg)

# Generate chart
plot_statistics_chart(stats_df, chart_jpg)

# Display the map here in notebook
from IPython.display import Image
Image(filename=str(map_jpg), width=600)